# VM.AI — Notebook trainer
**Colab notebook** — requires in /content/drive/MyDrive/VM.AI a clone of the most up-to-date repo.

---

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone or update the repo

In [ ]:
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Infiteri/VM.AI.git"
REPO_DIR = "/content/drive/MyDrive/VM.AI"

if os.path.exists(REPO_DIR):
    print("Repo already exists — pulling latest changes...")
    %cd {REPO_DIR}
    !git pull
else:
    print("Cloning repo...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print(f"Working directory: {os.getcwd()}")


## 3. Install dependencies

In [ ]:
!pip install -q transformers datasets huggingface_hub pyyaml numpy torch


## 4. Verify GPU
Make sure you have a GPU runtime: **Runtime → Change runtime type → T4 GPU**

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found — training will be very slow on CPU.")


## 5. Setup paths and import modules

In [ ]:
import sys, os

REPO_DIR = "/content/drive/MyDrive/VM.AI"
SRC_DIR  = os.path.join(REPO_DIR, "src")

sys.path.insert(0, SRC_DIR)
sys.path.insert(0, os.path.join(SRC_DIR, "parser"))

os.chdir(os.path.join(SRC_DIR, "parser"))
print("cwd:", os.getcwd())


In [ ]:
from cfg import EnvConfig

cfg = EnvConfig("colab")
print(f"Env           : {cfg.env}")
print(f"Max limit     : {cfg.max_limit}")
print(f"Epochs        : {cfg.num_train_epochs}")
print(f"fp16          : {cfg.fp16}")
print(f"Model cache   : {cfg.model_cache}")
print(f"Output dir    : {cfg.output_dir}")
print(f"Data path     : {cfg.data_path}")


## 6. Download base model (T5-small)
Cached to Drive — only downloads once.

In [ ]:
from huggingface_hub import snapshot_download

os.makedirs(cfg.model_cache, exist_ok=True)

if not os.path.exists(cfg.model_cache) or not os.listdir(cfg.model_cache):
    print("Downloading google/t5-small to Drive...")
    snapshot_download(repo_id="google-t5/t5-small", local_dir=cfg.model_cache)
else:
    print(f"Model already cached at {cfg.model_cache}")


## 7. Load data and generate dataset

In [ ]:
from yaml_parser import VMAI_YamlParser
from data_generator import VMAI_DataGenerator
import vars

parser = VMAI_YamlParser(cfg.data_path)
parser.load_yaml()
training_data = parser.parse()
print(f"Data loaded: {cfg.data_path}")

dataset = VMAI_DataGenerator(training_data).generate(cfg.max_limit)
split   = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split["train"]
test_dataset  = split["test"]

print(f"Train examples : {len(train_dataset)}")
print(f"Test examples  : {len(test_dataset)}")


## 8. Tokenize

In [ ]:
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg.model_cache)

def tokenize_function(examples):
    inputs = tokenizer(
        examples["input_text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    targets = tokenizer(
        examples["target_text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    labels = targets["input_ids"]
    labels = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels
    ]
    inputs["labels"] = np.array(labels, dtype=np.int64)
    return inputs

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test  = test_dataset.map(tokenize_function,  batched=True)

cols = ["input_ids", "attention_mask", "labels"]
tokenized_train.set_format(type="torch", columns=cols)
tokenized_test.set_format( type="torch", columns=cols)

print("Tokenization complete.")


## 9. Load model (resume if checkpoint exists)

In [ ]:
import torch
from transformers import T5ForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(cfg.output_dir, exist_ok=True)

is_resume = os.path.exists(cfg.output_dir) and os.listdir(cfg.output_dir)
if is_resume:
    print("Resuming from checkpoint...")
    model = T5ForConditionalGeneration.from_pretrained(cfg.output_dir)
else:
    print("Loading base T5-small...")
    model = T5ForConditionalGeneration.from_pretrained(cfg.model_cache)

model.to(device)
print(f"Model on: {device}")


## 10. Train

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

learning_rate = cfg.learning_rate_resume if is_resume else cfg.learning_rate_fresh

training_args = Seq2SeqTrainingArguments(
    output_dir=                  cfg.output_dir,
    eval_strategy=               "epoch",
    save_strategy=               "epoch",
    learning_rate=               learning_rate,
    weight_decay=                0.01,
    save_total_limit=            2,
    predict_with_generate=       True,
    push_to_hub=                 False,
    remove_unused_columns=       False,
    optim=                       "adafactor",
    num_train_epochs=            cfg.num_train_epochs,
    per_device_train_batch_size= cfg.per_device_train_batch_size,
    per_device_eval_batch_size=  cfg.per_device_eval_batch_size,
    gradient_accumulation_steps= cfg.gradient_accumulation_steps,
    fp16=                        cfg.fp16,
    dataloader_num_workers=      cfg.dataloader_num_workers,
    dataloader_pin_memory=       cfg.dataloader_pin_memory,
    logging_steps=               cfg.logging_steps,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=         model,
    args=          training_args,
    train_dataset= tokenized_train,
    eval_dataset=  tokenized_test,
    data_collator= data_collator,
)

print("Starting training...")
trainer.train()


## 11. Save model to Drive

In [ ]:
model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"✅ Model saved to {cfg.output_dir}")


## 12. Quick sanity test

In [ ]:
from transformers import T5ForConditionalGeneration, AutoTokenizer
import torch

model_test = T5ForConditionalGeneration.from_pretrained(cfg.output_dir).to(device)
tok_test   = AutoTokenizer.from_pretrained(cfg.output_dir)

test_input = "add: finish chemistry homework before Friday, pretty hard"
inputs = tok_test(test_input, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model_test.generate(**inputs, max_new_tokens=256)

result = tok_test.decode(output_ids[0], skip_special_tokens=True)
print(f"Input  : {test_input}")
print(f"Output : {result}")
